<a href="https://colab.research.google.com/github/AndresMontesDeOca/IIA/blob/main/4-exercise_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Andres Montes de Oca**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [ ]:
import sys
import os

# Add the correct path to 'aima_libs' within 'intro_ia'
repo_path = 'intro_ia/clase2/exercise'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print(f"Added {repo_path} to sys.path")

# Clone the repository containing the desired aima_libs package
!git clone https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/intro_ia.git

from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

Added intro_ia/clase2/exercise to sys.path
Cloning into 'intro_ia'...
remote: Enumerating objects: 2884, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 2884 (delta 77), reused 200 (delta 61), pack-reused 2656 (from 2)
Receiving objects: 100% (2884/2884), 626.76 MiB | 23.62 MiB/s, done.
Resolving deltas: 100% (1122/1122), done.
Updating files: 100% (310/310), done.


In [37]:
def search_algorithm(number_disks=5):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA #####

    # 1. Inicializamos la Frontera
    # Para DFS y BFS, podemos iniciar con una lista normal
    frontier = [NodeHanoi(initial_state)]

    # 2. Conjunto para guardar los estados visitados
    visited = set()

    # Variables para rastrear las métricas
    nodes_explored = 0
    max_depth = 0
    solution_found = False
    solution_node = None

    # 3. Bucle principal
    while len(frontier) > 0:

        # ===================================================================
        # ⚠️ DIFERENCIA CLAVE ENTRE DFS Y BFS ⚠️
        #
        # ACTUAL (DFS - Profundidad):
        # Usamos .pop() sin argumentos. Esto extrae el ÚLTIMO elemento de
        # la lista, comportándose como una Pila (LIFO).
        #
        # SI FUERA BFS (Anchura):
        # Cambiaríamos esta línea por: node = frontier.pop(0)
        # Esto extraería el PRIMER elemento, comportándose como una Cola (FIFO).
        # ===================================================================
        node = frontier.pop() # DFS
        # node = frontier.pop(0) # BFS

        # Evitamos re-explorar estados por los que ya pasamos
        state_rep = str(node.state)
        if state_rep in visited:
            continue

        visited.add(state_rep)
        nodes_explored += 1

        # Actualizamos la métrica de profundidad máxima
        current_depth = getattr(node, 'depth', 0)
        if current_depth > max_depth:
            max_depth = current_depth

        # ¿Llegamos a la meta?
        if problem.goal_test(node.state):
            solution_node = node
            solution_found = True
            break

        # Expandimos los hijos del nodo
        for next_node in node.expand(problem):
            # Solo los agregamos a la frontera si no los hemos visitado
            if str(next_node.state) not in visited:
                # ===========================================================
                # NOTA SOBRE BFS:
                # En BFS, para ahorrar memoria, suele ser más eficiente
                # comprobar si es la meta y agregarlo a 'visited' justo AQUÍ,
                # antes de hacer el .append(). En DFS, se hace al extraerlo.
                # ===========================================================
                frontier.append(next_node)

    # 4. Consolidamos las métricas requeridas
    metrics = {
        "solution_found": solution_found,
        "nodes_explored": nodes_explored,
        "states_visited": len(visited),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": float(getattr(solution_node, 'path_cost', current_depth)) if solution_found else 0.0,
    }

    solution = solution_node if solution_found else None

    ##### FIN DE ZONA EDITADA #####

    return solution, metrics

Se prueba la función:

In [38]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [39]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 122
states_visited: 122
nodes_in_frontier: 63
max_depth: 121
cost_total: 121.0


Veamos el camino de estados desde el principio a la solución:

In [32]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 2 | 1 | >
<Node HanoiState: 5 4 3 | 1 | 2>
<Node HanoiState: 5 4 3 |  | 2 1>
<Node HanoiState: 5 4 3 1 |  | 2>
<Node HanoiState: 5 4 3 1 | 2 | >
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 | 2 | 3 1>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 4 | 1 | 3 2>
<Node HanoiState: 5 4 2 | 1 | 3>
<Node HanoiState: 5 4 2 |  | 3 1>
<Node HanoiState: 5 4 2 1 |  | 3>
<Node HanoiState: 5 4 2 1 | 3 | >
<Node HanoiState: 5 4 2 | 3 | 1>
<Node HanoiState: 5 4 2 | 3 1 | >
<Node HanoiState: 5 4 | 3 1 | 2>
<Node HanoiState: 5 4 | 3 | 2 1>
<Node HanoiState: 5 4 1 | 3 | 2>
<Node HanoiState: 5 4 1 | 3 2 | >
<Node HanoiState: 5 4 | 3 2 | 1>
<Node HanoiState: 5 4 | 3 2 1 | >
<Node HanoiState: 5 | 3 2 1 | 4>
<Node HanoiState: 5 | 3 2 | 4 1>
<Node HanoiState: 5 1 | 3 2

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [33]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 4 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 3 from 2 to 3
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from